# Create CSV for PANDA_Subset_600

This notebook extracts information for your 600 slides from the full train.csv and creates a new CSV file with stratified k-fold splits.

In [ ]:
# Setup and Mount Google Drive
import os
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted!')
except:
    IN_COLAB = False
    print('Not in Colab environment')

Mounted at /content/drive
Google Drive mounted!


In [ ]:
# Configuration - UPDATE THESE PATHS IF NEEDED
CONFIG = {
    'base_folder': '/content/drive/MyDrive/PANDA_Subset_600',
    'full_csv_path': '/content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train.csv',
    'output_csv': '/content/drive/MyDrive/subset_600.csv',
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

Configuration:
  base_folder: /content/drive/MyDrive/PANDA_Subset_600
  full_csv_path: /content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/train.csv
  output_csv: /content/drive/MyDrive/subset_600.csv


In [ ]:
# Check folder structure and find actual slides
print('='*50)
print('CHECKING FOLDER STRUCTURE')
print('='*50)

base_folder = CONFIG['base_folder']
contents = os.listdir(base_folder)
print(f'Contents of PANDA_Subset_600: {contents[:10]}...' if len(contents) > 10 else f'Contents: {contents}')
print(f'Total items: {len(contents)}')

# Handle subfolder (images folder inside)
if 'images' in contents:
    subset_folder = os.path.join(base_folder, 'images')
    print(f'\n✓ Found "images" subfolder, using: {subset_folder}')
elif len(contents) == 1 and os.path.isdir(os.path.join(base_folder, contents[0])):
    subset_folder = os.path.join(base_folder, contents[0])
    print(f'\n✓ Found subfolder "{contents[0]}", using: {subset_folder}')
else:
    subset_folder = base_folder
    print(f'\n✓ Using base folder directly: {subset_folder}')

# List files in the actual folder
slide_files = os.listdir(subset_folder)
print(f'\nFiles in slide folder: {len(slide_files)}')
print(f'Sample files: {slide_files[:5]}')

CHECKING FOLDER STRUCTURE
Contents: ['images']
Total items: 1

✓ Found "images" subfolder, using: /content/drive/MyDrive/PANDA_Subset_600/images

Files in slide folder: 600
Sample files: ['02e7ef58dba5c2d67b6d3711dc081135.tiff', '03db76b0a1387cfb4206fd11f42051d7.tiff', '04875b8121c928cc022dd6f26a307d06.tiff', '0536843beb45cb6d5d7db5bf576a37af.tiff', '06636cdd43041e78141f2f5069fa62d5.tiff']


In [ ]:
# Step 1: Extract slide IDs from filenames
print('='*50)
print('STEP 1: EXTRACTING SLIDE IDs')
print('='*50)

slide_ids = []
for f in slide_files:
    if f.startswith('.'):  # Skip hidden files
        continue
    # Remove any extension (.tiff, .pt, .h5, etc.)
    slide_id = os.path.splitext(f)[0]
    if slide_id:
        slide_ids.append(slide_id)

slide_ids = list(set(slide_ids))  # Remove duplicates
print(f'✓ Found {len(slide_ids)} unique slide IDs')
print(f'\nSample IDs:')
for sid in slide_ids[:5]:
    print(f'  {sid}')

STEP 1: EXTRACTING SLIDE IDs
✓ Found 600 unique slide IDs

Sample IDs:
  41b5859aa0d162eb72387b3ee8a8903e
  cf385dd3211d78cdcfa8c90c2bab5b37
  4d24fcac403f5bea73a53bc6289d9044
  7c7189f32365e258a898eb268281337a
  490c134f47d483a880c183f85f66705e


In [ ]:
# Step 2: Load full train.csv
print('='*50)
print('STEP 2: LOADING FULL TRAIN.CSV')
print('='*50)

df_full = pd.read_csv(CONFIG['full_csv_path'])
print(f'✓ Full CSV has {len(df_full)} rows')
print(f'\nColumns: {list(df_full.columns)}')
print(f'\nGrade distribution in full dataset:')
print(df_full['isup_grade'].value_counts().sort_index())

STEP 2: LOADING FULL TRAIN.CSV
✓ Full CSV has 10616 rows

Columns: ['image_id', 'data_provider', 'isup_grade', 'gleason_score']

Grade distribution in full dataset:
isup_grade
0    2892
1    2666
2    1343
3    1242
4    1249
5    1224
Name: count, dtype: int64


In [ ]:
# Step 3: Filter to only the subset slides
print('='*50)
print('STEP 3: FILTERING TO SUBSET')
print('='*50)

df_subset = df_full[df_full['image_id'].isin(slide_ids)].copy()
print(f'✓ Matched {len(df_subset)} slides out of {len(slide_ids)}')

# Check for missing slides
matched_ids = set(df_subset['image_id'].tolist())
missing_ids = set(slide_ids) - matched_ids

if missing_ids:
    print(f'\n⚠ Warning: {len(missing_ids)} slides not found in train.csv')
    print(f'Missing samples: {list(missing_ids)[:10]}')
else:
    print('\n✓ All slides found in train.csv!')

STEP 3: FILTERING TO SUBSET
✓ Matched 600 slides out of 600

✓ All slides found in train.csv!


In [ ]:
# Step 4: Add stratified k-fold splits
print('='*50)
print('STEP 4: ADDING STRATIFIED K-FOLD')
print('='*50)

if len(df_subset) == 0:
    print('✗ ERROR: No slides matched! Check your paths.')
else:
    from sklearn.model_selection import StratifiedKFold

    df_subset['kfold'] = -1
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (_, val_idx) in enumerate(skf.split(df_subset, df_subset['isup_grade'])):
        df_subset.iloc[val_idx, df_subset.columns.get_loc('kfold')] = fold

    print('✓ Added 5-fold stratified splits')
    print(f'\nFold distribution:')
    print(df_subset['kfold'].value_counts().sort_index())

STEP 4: ADDING STRATIFIED K-FOLD
✓ Added 5-fold stratified splits

Fold distribution:
kfold
0    120
1    120
2    120
3    120
4    120
Name: count, dtype: int64


In [ ]:
# Step 5: Show grade distribution
print('='*50)
print('STEP 5: GRADE DISTRIBUTION')
print('='*50)

print('\nISUP Grade distribution:')
print(df_subset['isup_grade'].value_counts().sort_index())

print('\nData provider distribution:')
print(df_subset['data_provider'].value_counts())

# Check if balanced
grade_counts = df_subset['isup_grade'].value_counts().sort_index()
if grade_counts.min() >= 90 and grade_counts.max() <= 110:
    print('\n✓ Dataset appears balanced (~100 per grade)')
else:
    print(f'\n⚠ Dataset may not be balanced (min: {grade_counts.min()}, max: {grade_counts.max()})')

STEP 5: GRADE DISTRIBUTION

ISUP Grade distribution:
isup_grade
0    100
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64

Data provider distribution:
data_provider
radboud       325
karolinska    275
Name: count, dtype: int64

✓ Dataset appears balanced (~100 per grade)


In [ ]:
# Step 6: Save CSV
print('='*50)
print('STEP 6: SAVING CSV')
print('='*50)

output_path = CONFIG['output_csv']

# Create directory if needed
output_dir = os.path.dirname(output_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)

df_subset.to_csv(output_path, index=False)
print(f'✓ Saved to: {output_path}')
print(f'✓ Total rows: {len(df_subset)}')

# Verify file exists
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f'✓ File size: {file_size / 1024:.2f} KB')
else:
    print('✗ ERROR: File not saved!')

STEP 6: SAVING CSV
✓ Saved to: /content/drive/MyDrive/subset_600.csv
✓ Total rows: 600
✓ File size: 29.69 KB


In [ ]:
# Preview the saved CSV
print('='*50)
print('PREVIEW')
print('='*50)

print(f'\nColumns: {list(df_subset.columns)}')
print(f'\nFirst 10 rows:')
print(df_subset.head(10).to_string())

PREVIEW

Columns: ['image_id', 'data_provider', 'isup_grade', 'gleason_score', 'kfold']

First 10 rows:
                             image_id data_provider  isup_grade gleason_score  kfold
110  02e7ef58dba5c2d67b6d3711dc081135    karolinska           1           3+3      4
151  03db76b0a1387cfb4206fd11f42051d7    karolinska           1           3+3      2
173  04875b8121c928cc022dd6f26a307d06       radboud           5           5+5      3
196  0536843beb45cb6d5d7db5bf576a37af    karolinska           1           3+3      3
240  06636cdd43041e78141f2f5069fa62d5       radboud           4           3+5      2
265  0704a3d9638d1657d8b36968ce2b92f8    karolinska           1           3+3      2
271  0728601281f67ec7964014e1aa3caa35    karolinska           1           3+3      2
275  0748aba46621600d1e6b6ec02f90dfe5    karolinska           2           3+4      4
329  087fd9ebfe6fa82728be6c7eabe1d4e4    karolinska           3           4+3      4
338  08b04f43997889865568c2c0d7bba8ab       ra

In [ ]:
# Summary
print('='*50)
print('SUMMARY')
print('='*50)

print(f'''
CSV Created Successfully!

File: {CONFIG['output_csv']}
Total slides: {len(df_subset)}
Columns: {list(df_subset.columns)}

Grade Distribution:
{df_subset['isup_grade'].value_counts().sort_index().to_string()}

Ready to use with CLAM_TR_v16_Validation_Experiments.ipynb!
''')

SUMMARY

CSV Created Successfully!

File: /content/drive/MyDrive/subset_600.csv
Total slides: 600
Columns: ['image_id', 'data_provider', 'isup_grade', 'gleason_score', 'kfold']

Grade Distribution:
isup_grade
0    100
1    100
2    100
3    100
4    100
5    100

Ready to use with CLAM_TR_v16_Validation_Experiments.ipynb!

